# GenieX NPU Inference Examples

This notebook demonstrates how to use the GenieX for various AI inference tasks on NPU devices, including:

- **LLM (Large Language Model)**: Text generation and conversation
- **VLM (Vision Language Model)**: Multimodal understanding and generation

## Prerequisites

### 1. Install the correct Python version

If you prefer, we also offer a video tutorial for the installation. Check it out [here](https://www.youtube.com/watch?v=ziXKPRX0Ufo).

GenieX requires **Python 3.11 – 3.13 (ARM64 build)** on Windows ARM.
Please **download and install the official ARM64 Python** from the [python-3.11.1-arm64.exe](https://www.python.org/ftp/python/3.11.1/python-3.11.1-arm64.exe). Make sure you read the instructions below carefully before proceeding.

> ❗ **IMPORTANT**: Make sure you select "Add python.exe to PATH" on the first screen of the installation wizard.

> 🛑 Make sure you restart the terminal or your IDE after installation.

> ⚠️ Do **not** use Conda or x86 builds — they are incompatible with native ARM64 binaries. If you are in a conda environment, run `conda deactivate` first.

Verify the installation:

In case your environment path gets overriden by some environment manager, we recommend you to run the following commands to restore PATH variable from system settings.
```powershell
$systemPath = [Environment]::GetEnvironmentVariable('Path', 'Machine')
$userPath   = [Environment]::GetEnvironmentVariable('Path', 'User')
$env:Path   = "$userPath;$systemPath"
```

Then verify your python executable has the correct architecture and version (3.11 - 3.13)
```sh
python -c "import sys, platform; print(f'Python version: {sys.version}')"
```

Your output should look like:

> Python version: 3.11.0 (main, Oct 24 2022, 18:15:22) [MSC v.1933 64 bit (ARM64)]

Expected output must contain version `3.11.0` and architecture `ARM64`.

If it does show `AMD64` or incorrect version, try the following:

- (If you have conda installed) Run `conda deactivate` to deactivate the current conda environment.
- (If your `python` executable points to the x86 version) You may need to make the ARM64 Python come before the x86 Python in your PATH.
  - Hit the `Win` key, and type `env`, and hit Enter to select `Edit the system environment variables` setting.
  - Click on `Environment Variables...` button.
  - Select `Path` and click `Edit...`.
  - Find your ARM64 Python installation path, and move it to the top of the list.
  - Hit `OK` for several times to close all the dialogs and save the changes.
- (If you forgot to select "Add python.exe to PATH" on the first screen of the installation wizard)
  - Run the installation wizard again, follow the instructions to remove the current installation, and then reinstall from the Wizard. Make sure to select "Add python.exe to PATH" this time.

---

### 2. Create and activate a virtual environment

`cd` to the current project root directory `cd path/to/geniex`.

```sh
python -m venv geniex-env
geniex-env\Scripts\activate
```

---

### 3. Install GenieX

```bash
pip install -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple geniex
```

#### Troubleshooting: SSL / certificate errors (corporate proxy)

If the installation fails with an SSL certificate error (common on corporate networks with TLS inspection), you can pre-download the SDK manually and point the installer to it. Run the following **PowerShell** commands **before** `pip install`, replacing `<version>` with the version you are installing:

```powershell
$env:GENIEX_VERSION = "<version>"
$base = "https://qaihub-public-assets.s3.us-west-2.amazonaws.com/qai-hub-geniex/$env:GENIEX_VERSION"
$zip  = "geniex-sdk-windows-arm64-$env:GENIEX_VERSION.zip"

Invoke-WebRequest "$base/$zip"        -OutFile $zip
Invoke-WebRequest "$base/$zip.sha256" -OutFile "$zip.sha256"

$env:GENIEX_SDK_DOWNLOAD_URL = "file:///$($(Get-Location).Path.Replace('\','/'))"
pip install -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple geniex

Remove-Item $zip, "$zip.sha256"
Remove-Item Env:GENIEX_SDK_DOWNLOAD_URL
```

> 💡 `Invoke-WebRequest` uses the system's certificate store (including corporate CA certificates), so it can succeed where Python's SSL stack cannot. The SDK zip is downloaded to the current directory and automatically cleaned up after installation.

---

### 4. Select the venv as your Jupyter Notebook kernel

- Depending the editor you are using, the way to change kernel might be different. For Cursor / VS Code, they are located at the top right corner of you code window.
- Look for and select the `geniex-env`, or the custom virtual environment you have created. The kernel should automatically reload in most IDEs.


### Verification of Kernel

Run the following code to ensure you have the right kernel running.


In [ ]:
import sys
import platform

# ANSI color codes
RED = "\033[91m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
BOLD = "\033[1m"
RESET = "\033[0m"

min_ver = (3, 11)
max_ver = (3, 13)
current_ver = sys.version_info
arch = platform.machine()

if not (min_ver <= (current_ver.major, current_ver.minor) < max_ver) or arch.lower() != "arm64":
    print("\n" + "=" * 80)
    print(f"{BOLD}{RED}WARNING: Your Python version or architecture is not compatible.{RESET}")
    print(f"Detected version: {current_ver.major}.{current_ver.minor}, architecture: {arch}")
    print(f"{YELLOW}Required: Python 3.11 - 3.13 & architecture 'arm64'.{RESET}")
    print("=" * 80)
    print(f"{RED}DO NOT continue to the following code!{RESET}\n")
    print("To install arm64 Python:")
    print("  - Download Python 3.11-3.13 for arm64 from https://www.python.org/downloads/")
    print("  - Install and verify by running: python3 --version and python3 -c 'import platform; print(platform.machine())'")
    print("  - Launch Jupyter and make sure to select the arm64 Python kernel in 'Kernel > Change kernel'.")
    sys.exit(1)
else:
    print(f"{GREEN}[VERIFICATION PASSED] Python version and architecture are correct. You may continue to the following sections.{RESET}")


## 1. LLM (Large Language Model) NPU Inference

In [ ]:
from geniex import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B-GGUF", # HF repo id, or local GGUF/.bin path
    device_map="auto",      # "auto" | "<plugin>" | "<plugin>:<device>"
)

messages = [{"role": "user", "content": "What is 2+2?"}]
prompt = model.tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
)

# One-shot
output = model.generate(prompt, max_new_tokens=256)
print(output.text)
print(f"[{output.profile.generated_tokens} tok, "
      f"{output.profile.decode_speed:.1f} tok/s, stop={output.profile.stop_reason}]")

# Streaming
streamer = model.generate(prompt, max_new_tokens=256, stream=True)
for chunk in streamer:
    print(chunk, end="", flush=True)

model.close()

## 2. VLM (Vision Language Model) NPU Inference

First, download a sample image:


In [ ]:
import os
from geniex import AutoModelForVision2Seq

# FIXME: replace with your image path
image_path = os.path.abspath("<path_to_your_image>")

# Optionally, replace with a local .gguf / .bin path
model = AutoModelForVision2Seq.from_pretrained("ggml-org/SmolVLM-500M-Instruct-GGUF", quant="Q8_0", device_map="llama_cpp")
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image_path},
        {"type": "text", "text": "Describe the image."},
    ],
}]
prompt = model.tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
)
print(model.generate(prompt, images=[image_path]).text)